In [18]:
import duckdb
import os
import pandas as pd

In [19]:
CURRENT_DIR = os.getcwd()
while not CURRENT_DIR.endswith("apple-retail-pipeline") and CURRENT_DIR != "/":
    CURRENT_DIR = os.path.dirname(CURRENT_DIR)
os.chdir(CURRENT_DIR)

WAREHOUSE_DIR = os.path.join(CURRENT_DIR, "data", "warehouse", "apple")

DB_PATH = os.path.join(WAREHOUSE_DIR, "apple_warehouse.duckdb")
con = duckdb.connect(DB_PATH)

In [20]:
tables = {
    "fact_sales": "fact_sales.parquet",
    "dim_store": "dim_store.parquet",
    "dim_product": "dim_product.parquet",
    "dim_category": "dim_category.parquet",
    "dim_warranty": "dim_warranty.parquet",
    "dim_date": "dim_date.parquet",
}

In [21]:
for name, fname in tables.items():
    path = os.path.join(WAREHOUSE_DIR, fname)
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM parquet_scan('{path}')")
print("All tables registered as views in DuckDB.\n")

All tables registered as views in DuckDB.



In [22]:
print("TOP 10 STORES WITH THE MOST REVENUE")
print(con.execute("""
SELECT s.store_name, SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_store s ON f.store_id = s.store_id
GROUP BY s.store_name
ORDER BY total_revenue DESC
LIMIT 10;
""").df(), "\n")

TOP 10 STORES WITH THE MOST REVENUE
             store_name  total_revenue
0       Apple Chadstone    165220023.0
1   Apple Covent Garden    165176160.0
2  Apple The Dubai Mall    163741158.0
3    Apple Orchard Road    162709375.0
4   Apple Central World    162552209.0
5  Apple Champs-Elysees    161720430.0
6      Apple Dubai Mall     84478026.0
7       Apple Southland     83830993.0
8           Apple Kyoto     83706593.0
9         Apple Fukuoka     83576408.0 



In [25]:
print("AVERAGE REVENUE GROUP BY STORES")
print(con.execute("""
SELECT m.store_name, AVG(n.revenue) AS avg_revenue
FROM dim_store AS m
JOIN fact_sales AS n
ON m.store_id = n.store_id
GROUP BY m.store_name
""").df(), "\n")

AVERAGE REVENUE GROUP BY STORES
                     store_name  avg_revenue
0                 Apple Fukuoka  5940.887688
1               Apple Chadstone  5932.283329
2   Apple North Michigan Avenue  5904.908786
3            Apple Eaton Centre  5921.165003
4           Apple Grand Central  5895.436552
..                          ...          ...
64             Apple Omotesando  5969.742939
65               Apple Yorkdale  5938.214347
66               Apple Santa Fe  5950.303629
67          Apple Walnut Street  5946.688159
68              Apple Ala Moana  5924.299273

[69 rows x 2 columns] 



In [ ]:
print("COUNT OF SALED CATEGORIES PER STORES")
print(con.execute("""
SELECT 
    n.category_id, 
    COUNT(DISTINCT m.store_id) AS distinct_stores
FROM fact_sales AS m
JOIN dim_product AS n 
    ON m.product_id = n.product_id
JOIN dim_store AS k 
    ON m.store_id = k.store_id
GROUP BY n.category_id
ORDER BY distinct_stores DESC;
""").df(), "\n")


COUNT OF SALED CATEGORIES PER STORES
  category_id  distinct_stores
0       CAT-1               75
1      CAT-10               75
2       CAT-2               75
3       CAT-9               75
4       CAT-3               75
5       CAT-7               75
6       CAT-4               75
7       CAT-6               75
8       CAT-5               75
9       CAT-8               75 



In [33]:
print("SUM AND AVERAGES OF ALL CATEGORIES")
print(con.execute("""
SELECT category_name, SUM(revenue) AS sum_revenue, AVG(revenue) AS avg_revenue 
FROM fact_sales AS m
JOIN dim_product AS n
ON m.product_id = n.product_id
JOIN dim_category AS k
ON n.category_id = k.category_id
GROUP BY category_name
ORDER BY sum_revenue DESC
""").df(), "\n")


SUM AND AVERAGES OF ALL CATEGORIES
          category_name  sum_revenue  avg_revenue
0                Tablet  953443623.0  8139.212434
1           Accessories  927115953.0  5658.355883
2            Smartphone  865147932.0  5705.425704
3                 Audio  794980579.0  6175.517777
4                Laptop  763382551.0  6560.016422
5              Wearable  667537447.0  6326.109940
6               Desktop  538481345.0  4609.811877
7  Subscription Service  368463489.0  4517.643101
8      Streaming Device  191622603.0  5472.742417
9         Smart Speaker   96117508.0  4078.824867 



In [40]:
print("AVERAGE REVENUE GROUP BY STORES PER CATEGORIES")
print(con.execute("""
SELECT k.category_name, c.store_name, ROUND(AVG(revenue),2) AS avg_revenue
FROM fact_sales AS m
JOIN dim_product AS n
ON m.product_id = n.product_id
JOIN dim_category AS k
ON n.category_id = k.category_id
JOIN dim_store AS c
ON c.store_id = m.store_id
GROUP BY category_name, store_name
ORDER BY k.category_name, avg_revenue DESC;
""").df(), "\n")


AVERAGE REVENUE GROUP BY STORES PER CATEGORIES
    category_name                    store_name  avg_revenue
0     Accessories            Apple Nanjing East      5880.12
1     Accessories  Apple The Americana at Brand      5831.25
2     Accessories         Apple Michigan Avenue      5783.49
3     Accessories           Apple Schildergasse      5768.85
4     Accessories           Apple Pioneer Place      5768.67
..            ...                           ...          ...
685      Wearable    Apple Jewel Changi Airport      6145.89
686      Wearable           Apple Rideau Centre      6103.52
687      Wearable                Apple Shinjuku      6049.31
688      Wearable                  Apple Antara      6031.01
689      Wearable        Apple Parque La Colina      6008.80

[690 rows x 3 columns] 



In [ ]:
print("TOP SELLING CATEGORY PER STORE")
print(con.execute("""
WITH category_revenue AS (
    SELECT 
        c.store_name,
        k.category_name,
        SUM(m.revenue) AS total_revenue
    FROM fact_sales AS m
    JOIN dim_product AS n
        ON m.product_id = n.product_id
    JOIN dim_category AS k
        ON k.category_id = n.category_id
    JOIN dim_store AS c
        ON c.store_id = m.store_id
    GROUP BY c.store_name, k.category_name
)
SELECT store_name, category_name, total_revenue
FROM (
    SELECT 
        store_name,
        category_name,
        total_revenue,
        ROW_NUMBER() OVER(PARTITION BY store_name ORDER BY total_revenue DESC) AS rank
    FROM category_revenue
)
WHERE rank = 1
ORDER BY total_revenue DESC;
""").df(), "\n")

TOP SELLING CATEGORY PER STORE
              store_name category_name  total_revenue
0        Apple Chadstone        Tablet     25847210.0
1    Apple Covent Garden        Tablet     25617576.0
2   Apple The Dubai Mall        Tablet     25509024.0
3     Apple Orchard Road   Accessories     25180626.0
4   Apple Champs-Elysees        Tablet     24641122.0
..                   ...           ...            ...
64          Apple Sydney        Tablet     12376140.0
65          Apple Andino        Tablet     12313171.0
66   Apple Grand Central   Accessories     12209864.0
67    Apple Causeway Bay        Tablet     12082777.0
68          Apple Antara    Smartphone     12045658.0

[69 rows x 3 columns] 

